<div class="align-center">
<a href="https://oumi.ai/"><img src="https://oumi.ai/docs/en/latest/_static/logo/header_logo.png" height="200"></a>

[![Documentation](https://img.shields.io/badge/Documentation-latest-blue.svg)](https://oumi.ai/docs/en/latest/index.html)
[![Discord](https://img.shields.io/discord/1286348126797430814?label=Discord)](https://discord.gg/oumi)
[![GitHub Repo stars](https://img.shields.io/github/stars/oumi-ai/oumi)](https://github.com/oumi-ai/oumi)
</div>

👋 Welcome to Open Universal Machine Intelligence (Oumi)!

🚀 Oumi is a fully open-source platform that streamlines the entire lifecycle of foundation models — from [data preparation](https://oumi.ai/docs/en/latest/resources/datasets/datasets.html) and [training](https://oumi.ai/docs/en/latest/user_guides/train/train.html) to [evaluation](https://oumi.ai/docs/en/latest/user_guides/evaluate/evaluate.html) and [deployment](https://oumi.ai/docs/en/latest/user_guides/launch/launch.html).

🤝 Make sure to join our [Discord community](https://discord.gg/oumi) to get help, share your experiences, and contribute to the project!

⭐ If you like Oumi and you would like to support it, please give it a star on [GitHub](https://github.com/oumi-ai/oumi).

# AIDE Reward Function Design

In the [GRPO Letter Counting notebook](https://github.com/oumi-ai/oumi/blob/main/notebooks/Oumi%20-%20Train%20a%20Letter%20Counting%20Model%20using%20GRPO.ipynb), we trained Llama 3.2 3B to count letters using GRPO with a hand-written reward function. The reward was simple: `-abs(predicted_count - target_count)`. It worked — but designing reward functions is one of the hardest parts of RLHF. What if an AI could design a better one?

In this notebook, we use [AIDE (AI-Driven Exploration)](https://github.com/WecoAI/aideml) to **autonomously generate and iterate on reward functions**. AIDE's tree-search algorithm will:
1. Draft candidate reward functions from scratch
2. Test each by running a short GRPO training
3. Debug failures and improve the best candidates
4. Return the highest-performing reward function

This notebook focuses on the **REWARD_FUNCTION** surface. AIDE supports multiple surfaces:

| Surface | Notebook | Description |
|---------|----------|-------------|
| CONFIG_SEARCH | [AIDE Agentic Optimization](Oumi%20-%20AIDE%20Agentic%20Optimization.ipynb) | Optimize training hyperparameters |
| **REWARD_FUNCTION** | **This notebook** | Design reward functions for GRPO/RLHF |
| EVAL_FUNCTION | [AIDE Custom Evaluation](Oumi%20-%20AIDE%20Custom%20Evaluation.ipynb) | Generate evaluation functions |
| FULL_PIPELINE | Coming soon | Generate complete training scripts |

## Prerequisites

### Machine Requirements

GRPO training requires significant GPU memory. For Llama 3.2 3B, you need ~40GB VRAM. For a lighter test, you can use SmolLM 135M with ~8GB VRAM.

### LLM Access

AIDE needs an LLM to generate and evaluate code. Serve a model locally (free) or use a cloud API.

```bash
# Local model (recommended):
vllm serve Qwen/Qwen3-Coder-Next --port 8000
```

In [ ]:
%pip install uv
!cd .. && uv pip install -e ".[aide]"

# For production: !uv pip install "oumi[aide]"

In [ ]:
import os
from pathlib import Path

# === LLM Access (uncomment one) ===
# os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"  # Local model
# os.environ["OPENAI_API_KEY"] = "dummy"
# os.environ["OPENAI_API_KEY"] = ""      # OpenAI
# os.environ["ANTHROPIC_API_KEY"] = ""   # Anthropic
# os.environ["GEMINI_API_KEY"] = ""      # Gemini

tutorial_dir = "aide_reward_tutorial"
Path(tutorial_dir).mkdir(parents=True, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Auto-detect LLM provider.
has_local = bool(os.environ.get("OPENAI_BASE_URL"))
has_openai = bool(os.environ.get("OPENAI_API_KEY")) and not has_local
has_anthropic = bool(os.environ.get("ANTHROPIC_API_KEY"))
has_gemini = bool(os.environ.get("GEMINI_API_KEY"))

if has_local:
    CODE_MODEL = FEEDBACK_MODEL = "Qwen/Qwen3-Coder-Next"
    provider = "Local server"
elif has_openai:
    CODE_MODEL, FEEDBACK_MODEL = "o4-mini", "gpt-5-mini"
    provider = "OpenAI"
elif has_anthropic:
    CODE_MODEL = FEEDBACK_MODEL = "claude-sonnet-4-20250514"
    provider = "Anthropic"
elif has_gemini:
    CODE_MODEL = FEEDBACK_MODEL = "gemini-2.5-flash"
    provider = "Gemini"
else:
    CODE_MODEL, FEEDBACK_MODEL = "o4-mini", "gpt-5-mini"
    provider = None

print(f"Provider: {provider or 'None (will show example output)'}")

## The Existing Reward Function

Oumi ships a hand-written reward function for letter counting at `src/oumi/datasets/grpo/rewards/count_letters_rewards.py`. It extracts the model's predicted count from `\\boxed{}` format and computes `-abs(predicted - target)`. Let's look at it:

In [ ]:
import inspect

from oumi.datasets.grpo.rewards.count_letters_rewards import (
    count_letters,  # type: ignore
)

print(inspect.getsource(count_letters))

## The Dataset

The letter counting task uses `oumi-ai/oumi-letter-count`. Each example asks the model to count occurrences of a letter in a word. Let's look at a few samples:

In [ ]:
from pprint import pprint

from oumi.datasets.grpo.letter_count import LetterCountGrpoDataset  # type: ignore

dataset = LetterCountGrpoDataset(
    dataset="oumi-ai/oumi-letter-count-clean", split="validation"
)
print(f"Dataset size: {len(dataset)} examples\n")
print("Sample prompt:")
pprint(dataset.conversation(0).to_dict())

### What Makes a Good Reward Function?

A reward function for GRPO must follow this signature:

```python
def reward_fn(
    completions: list[str],
    prompts: list[str] | None = None,
    **kwargs,
) -> list[float]
```

Key design considerations:
- **Correct answers** should get the highest reward
- **Partial credit** for close answers can help learning
- **Format compliance** (using `\\boxed{}`) can be rewarded separately
- **Brevity vs verbosity** trade-offs matter for efficiency

The hand-written version above is simple. Can AIDE design something more sophisticated?

### Testing the Existing Reward

Let's see how the hand-written reward scores different types of responses. This gives us a baseline to compare AIDE's reward against:

In [ ]:
from oumi.datasets.grpo.rewards.count_letters_rewards import (  # type: ignore
    compute_letter_count_reward,
)

# The word "STRAWBERRY" has 3 R's. Test various model responses:
target_count = 3
test_cases = [
    (
        "Correct with format",
        r"The letter R appears \boxed{3} times.",
        3,
    ),
    ("Off by one", r"I count \boxed{2} R's.", 2),
    ("Way off", r"There are \boxed{7} R's.", 7),
    ("No boxed format", "I think there are 3 R's.", None),
    ("Empty response", "", None),
]

print("Existing reward function scores:")
print(f"{'Response type':<25} {'Count':>5} {'Reward':>8}")
print("-" * 42)
for label, response, expected_count in test_cases:
    reward = compute_letter_count_reward(response, target_count)
    print(f"{label:<25} {str(expected_count):>5} {reward:>8.3f}")

## Let AIDE Design a Reward Function

We configure AIDE with `surface=REWARD_FUNCTION` and give it a natural language goal describing what the reward should optimize for. AIDE's `test_reward()` helper handles:
- Registering the reward function with Oumi's registry
- Running a short GRPO training to test it
- Extracting the metric

In [ ]:
from oumi.core.configs import (  # type: ignore
    AideConfig,
    AideParams,
    AideSearchParams,
    ModelParams,
)
from oumi.core.configs.params.aide_params import (  # type: ignore
    AideExecParams,
    AideLLMParams,
    AideOptimizationSurface,
)

config = AideConfig(
    model=ModelParams(
        model_name="meta-llama/Llama-3.2-3B-Instruct",
        torch_dtype_str="bfloat16",
        trust_remote_code=True,
    ),
    goal=(
        "Design a reward function for a letter counting task. "
        "The model receives prompts like 'How many times does the letter R "
        "appear in STRAWBERRY?' and must answer with a number in \\boxed{} format. "
        "The reward should give higher scores for correct counts, partial credit "
        "for close answers, and a small bonus for using the correct format."
    ),
    base_training_config="configs/examples/letter_counting/grpo/train.yaml",
    aide=AideParams(
        steps=5,
        surface=AideOptimizationSurface.REWARD_FUNCTION,
        target_metric="reward_accuracy",
        target_direction="maximize",
        output_dir=f"{tutorial_dir}/output",
        workspace_dir=f"{tutorial_dir}/workspace",
        code_llm=AideLLMParams(model=CODE_MODEL, temperature=0.5),
        feedback_llm=AideLLMParams(model=FEEDBACK_MODEL, temperature=0.5),
        search=AideSearchParams(num_drafts=2),
        execution=AideExecParams(timeout=300),
    ),
)

print(f"Surface:    {config.aide.surface.value}")
print(f"Metric:     {config.aide.target_metric} ({config.aide.target_direction})")
print(f"Code LLM:   {config.aide.code_llm.model}")
print(f"Steps:      {config.aide.steps}")

In [ ]:
from oumi.aide import aide as run_aide  # type: ignore

if provider:
    result = run_aide(config, verbose=True)

    print(f"\nBest metric: {result.best_metric}")
    print(f"Good: {result.good_solutions}, Buggy: {result.buggy_solutions}")
    print(f"Best solution: {result.best_solution_path}")
else:
    print("[Skipped \u2014 no LLM provider configured]")
    print("\nExample: AIDE designs a reward function that gives:")
    print("  1.0 for exact correct answer in \\boxed{} format")
    print("  0.5 for answer within \u00b11 of correct")
    print("  0.2 for correct format but wrong answer")
    print("  0.0 for no valid answer")

## Understanding the Results

AIDE's output includes:
- **best_code**: The Python source of the best reward function found
- **best_metric**: How well the reward performed during GRPO training
- **journal_path**: A JSON file with every candidate tried, useful for analysis

If all solutions were buggy, increase `steps` or check the journal for common errors. AIDE's debug phase should fix most issues automatically.

In [ ]:
# Show the best reward function code (if available)
if provider and result.best_code:
    print("--- AIDE's best reward function ---")
    print(result.best_code[:2000])
    print("---")

## Summary

In this notebook, you learned how to:

1. ✅ Use AIDE's REWARD_FUNCTION surface to auto-generate reward functions
2. ✅ Understand the reward function signature for GRPO
3. ✅ Compare AIDE-designed rewards with hand-written ones
4. ✅ Analyze results from the AIDE search journal

## References & Credits

- **AIDE** by [Weco AI](https://www.weco.ai/) — [GitHub](https://github.com/WecoAI/aideml) | [Paper](https://arxiv.org/abs/2502.13138)
- **Oumi** — [Website](https://oumi.ai/) | [GitHub](https://github.com/oumi-ai/oumi) | [Docs](https://oumi.ai/docs/)

# 🧭 What's Next?

Congrats on finishing this notebook! Feel free to check out our other [notebooks](https://github.com/oumi-ai/oumi/tree/main/notebooks) in the [Oumi GitHub](https://github.com/oumi-ai/oumi), and give us a star! You can also join the Oumi community over on [Discord](https://discord.gg/oumi).

- **[AIDE Agentic Optimization](Oumi%20-%20AIDE%20Agentic%20Optimization.ipynb)** — CONFIG_SEARCH surface: optimize hyperparameters
- **[AIDE Custom Evaluation](Oumi%20-%20AIDE%20Custom%20Evaluation.ipynb)** — EVAL_FUNCTION surface: auto-generate evaluation functions
- **[GRPO Letter Counting](Oumi%20-%20Train%20a%20Letter%20Counting%20Model%20using%20GRPO.ipynb)** — The original GRPO notebook this builds on
- **[Finetuning Tutorial](Oumi%20-%20Finetuning%20Tutorial.ipynb)** — LoRA-tune models with Oumi

Happy optimizing!